# 🛒 Historical Full Load Fact Pipeline (`fact_orders` & `sb_fact_orders`)
This notebook executes the complete historical backfill pipeline for the **Sales Orders Fact Tables**:
* **Bronze:** Ingests raw landed CSV orders with explicit schemas and Change Data Feed, then archives files to processed storage.
* **Silver:** Cleanses quantities, normalizes multi-format date strings, broadcasts product dimensions to resolve surrogate key `product_code`, and merges into `silver.orders`.
* **Gold Subsidiary (`sb_fact_orders`):** Persists daily operational sales orders at daily transaction grain.
* **Gold Enterprise (`fact_orders`):** Rolls up daily transactions to monthly corporate grain `(date, product_code, customer_code)`, validates data quality, merges into parent fact, and optimizes storage via Z-ORDER.

### 📌 Step 1: Import Core PySpark & Delta Lake Libraries
* **Purpose:** Loads necessary PySpark SQL analytical functions and Delta Lake table abstractions required for high-volume transactional processing and Lakehouse ACID merges.
* **Logic & Transformations:** Imports `pyspark.sql.functions as F` and `DeltaTable` from `delta.tables`.
* **Inputs & Dependencies:** PySpark runtime and `delta-spark` package.
* **Outputs & Medallion State:** Module namespaces `F` and `DeltaTable` available in session scope.

In [1]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### 📌 Step 2: Runtime Bootstrap & Project Utilities Execution
* **Purpose:** Configures modular Python paths and executes shared project utilities to establish active environment configurations, conformed schemas, and audit tools.
* **Logic & Transformations:** Resolves repository root path on `sys.path`, executes `%run ./utilities`, and initializes Databricks compatibility shims.
* **Inputs & Dependencies:** Shared Lakehouse utilities (`./utilities.py` / `utilities.ipynb`).
* **Outputs & Medallion State:** Pre-populated `spark`, `dbutils`, `display`, and global configurations in session scope.

In [2]:
# Initialize environment & Databricks compatibility (noop in Databricks)
import sys, os
try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()
repo_root = os.path.abspath(os.path.join(current_dir, "..")) if os.path.basename(current_dir) in ["1_setup", "2_dimension_data_processing", "3_fact_dat_processing"] else current_dir
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.compat import init_notebook_context
spark, dbutils, display = init_notebook_context(globals())

# Load environment config, schemas, and utilities via relative path
%run ../1_setup/utilities


26/09/17 12:43:45 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


### 📌 Step 3: Verify Active Medallion Schema Configurations
* **Purpose:** Confirms that the target Lakehouse schemas (`bronze`, `silver`, `gold`) are properly defined and aligned with the active environment.
* **Logic & Transformations:** Prints `bronze_schema`, `silver_schema`, and `gold_schema` strings to standard output for visual verification.
* **Inputs & Dependencies:** Configuration variables exported by utilities in Step 2.
* **Outputs & Medallion State:** Schema names printed to cell output.

In [3]:
print(bronze_schema, silver_schema, gold_schema)

bronze silver gold


### 📌 Step 4: Pipeline Parameterization & Storage URI Construction
* **Purpose:** Establishes configurable parameters (`catalog`, `data_source` = `orders`) and derives storage URIs for the raw orders landing zone and archive directories.
* **Logic & Transformations:** Defines interactive Databricks text widgets, resolves active catalog and dataset name, and forms S3 paths (`base_path`, `landing_path`, `processed_path`) and table names (`bronze_table`, `silver_table`, `gold_table`).
* **Inputs & Dependencies:** Databricks widget inputs (`catalog`: `fmcg`, `data_source`: `orders`).
* **Outputs & Medallion State:** Pipeline URI paths and target table names registered.

In [4]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://spartsbar-2355/{data_source}'
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"
print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)


# define the tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

Base Path:  s3://spartsbar-2355/orders
Landing Path:  s3://spartsbar-2355/orders/landing/
Processed Path:  s3://spartsbar-2355/orders/processed/


### 📌 Step 5: Schema-Enforced Historical Backfill Ingestion from S3
* **Purpose:** Ingests raw historical orders CSV landing files using explicit schema enforcement, capturing ingestion audit metadata.
* **Logic & Transformations:** Binds explicit `orders_schema`, appends `current_timestamp()` as `read_timestamp`, and unpacks `_metadata.file_name` and `_metadata.file_size`.
* **Inputs & Dependencies:** Raw CSV files at `landing_path`.
* **Outputs & Medallion State:** Raw DataFrame `df` containing source order records with ingestion metadata.

In [5]:
df = (
    spark.read.format("csv")
        .option("header", True)
        .schema(orders_schema)  # Explicit StructType schema eliminates inferSchema scan
        .load(f"{landing_path}/*.csv")
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)
print("Total Ingested Rows: ", df.count())
df.show(5)
df.write.format("delta").option("delta.enableChangeDataFeed", "true").mode("append").saveAsTable(bronze_table)


26/09/17 12:43:45 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3://spartsbar-2355/orders/landing//*.csv.
org.apache.hadoop.fs.UnsupportedFileSystemException: No FileSystem for scheme "s3"
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3586)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3617)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3721)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3672)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:558)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:373)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:57)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSou

[Local Spark Emulation] S3 path detected without AWS credentials. Providing mock data for: s3://spartsbar-2355/orders/landing//*.csv


Total Ingested Rows:  4
+--------+--------------------+-----------+----------+---------+--------------------+--------------------+---------------+---------+
|order_id|order_placement_date|customer_id|product_id|order_qty|           _metadata|      read_timestamp|      file_name|file_size|
+--------+--------------------+-----------+----------+---------+--------------------+--------------------+---------------+---------+
|  ORD001|Tuesday, July 01,...|       1001|      P101|       50|{sample_data.csv,...|2026-09-17 12:43:...|sample_data.csv|     1024|
|  ORD002|          2025-07-15|       1002|      P102|       20|{sample_data.csv,...|2026-09-17 12:43:...|sample_data.csv|     1024|
|  ORD003|          01/08/2025|       1003|      P103|      100|{sample_data.csv,...|2026-09-17 12:43:...|sample_data.csv|     1024|
|  ORD004|     August 20, 2025|       1001|      P104|       15|{sample_data.csv,...|2026-09-17 12:43:...|sample_data.csv|     1024|
+--------+--------------------+-----------+--

26/09/17 12:43:47 ERROR Utils: Aborting task
org.apache.spark.sql.delta.DeltaAnalysisException: [DELTA_CREATE_TABLE_WITH_NON_EMPTY_LOCATION] Cannot create table ('`bronze`.`orders`'). The associated location ('file:/Users/nithin/Projects/Atlikon_DE/spark-warehouse/bronze.db/orders') is not empty and also not a Delta table.
	at org.apache.spark.sql.delta.DeltaErrorsBase.createTableWithNonEmptyLocation(DeltaErrors.scala:3386)
	at org.apache.spark.sql.delta.DeltaErrorsBase.createTableWithNonEmptyLocation$(DeltaErrors.scala:3385)
	at org.apache.spark.sql.delta.DeltaErrors$.createTableWithNonEmptyLocation(DeltaErrors.scala:4266)
	at org.apache.spark.sql.delta.commands.CreateDeltaTableCommand.assertPathEmpty(CreateDeltaTableCommand.scala:566)
	at org.apache.spark.sql.delta.commands.CreateDeltaTableCommand.checkPathEmpty$1(CreateDeltaTableCommand.scala:187)
	at org.apache.spark.sql.delta.commands.CreateDeltaTableCommand.$anonfun$handleCommit$1(CreateDeltaTableCommand.scala:205)
	at org.apache

### 📌 Step 6: Validate Ingested Record Volume & Sample Records
* **Purpose:** Reports total row count and renders sample records to verify ingestion completeness before committing to Bronze Delta storage.
* **Logic & Transformations:** Calls `df.count()` and prints first 5 rows.
* **Inputs & Dependencies:** Raw DataFrame `df`.
* **Outputs & Medallion State:** Total row count and sample preview printed to output.

### 📌 Step 7: Persist Historical Ingestion into Bronze Delta Table
* **Purpose:** Saves raw orders into immutable Bronze Delta Lake table with Change Data Feed enabled for auditability.
* **Logic & Transformations:** Writes `df` with `format('delta')`, sets `delta.enableChangeDataFeed = true`, and saves in `append` mode to `{bronze_table}`.
* **Inputs & Dependencies:** Ingested DataFrame `df`.
* **Outputs & Medallion State:** Bronze Delta table `fmcg.bronze.orders` appended on Delta Lake storage.

In [6]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("append") \
 .saveAsTable(bronze_table)

26/09/17 12:43:55 WARN CreateNamespaceExec: Namespace bronze was created concurrently. Ignoring.


[Local Spark Emulation] Adapted target table: fmcg.bronze.orders -> bronze.orders


### 📌 Step 8: File Archival: Move Landed Files to Processed Directory
* **Purpose:** Archives ingested CSV files from the landing directory to the processed directory, preventing accidental duplicate re-processing in subsequent pipeline runs.
* **Logic & Transformations:** Lists files via `dbutils.fs.ls(landing_path)` and iterates moving each file to `processed_path` using `dbutils.fs.mv()`.
* **Inputs & Dependencies:** Files at `landing_path`.
* **Outputs & Medallion State:** Landing directory cleared; raw files moved to `processed_path` archive.

In [7]:
files = dbutils.fs.ls(landing_path)
for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )

[Local dbutils.fs] ls('s3://spartsbar-2355/orders/landing/')


### 📌 Step 9: Query Bronze Orders to Initialize Silver Cleansing
* **Purpose:** Reads raw records back from Bronze Delta storage to isolate raw landing from transformation and conformance logic.
* **Logic & Transformations:** Executes `spark.sql(SELECT * FROM {bronze_table})` into memory.
* **Inputs & Dependencies:** Bronze Delta table `fmcg.bronze.orders`.
* **Outputs & Medallion State:** DataFrame `df_orders` loaded into session scope.

In [8]:
df_orders = spark.sql(f"SELECT * FROM {bronze_table}")
df_orders.show(2)

[Local Spark Emulation] Multi-part namespace adapted: SELECT * FROM fmcg.bronze.orders -> SELECT * FROM bronze.orders


+--------+--------------------+-----------+----------+---------+--------------------+--------------------+---------------+---------+
|order_id|order_placement_date|customer_id|product_id|order_qty|           _metadata|      read_timestamp|      file_name|file_size|
+--------+--------------------+-----------+----------+---------+--------------------+--------------------+---------------+---------+
|  ORD001|Tuesday, July 01,...|       1001|      P101|       50|{sample_data.csv,...|2026-09-17 12:43:...|sample_data.csv|     1024|
|  ORD002|          2025-07-15|       1002|      P102|       20|{sample_data.csv,...|2026-09-17 12:43:...|sample_data.csv|     1024|
+--------+--------------------+-----------+----------+---------+--------------------+--------------------+---------------+---------+
only showing top 2 rows


### 📌 Step 10: Multi-Format Date Normalization & Sentinel Cleansing
* **Purpose:** Sanitizes messy transactional fields: filters non-positive quantities, strips weekday text prefixes (`Tuesday, `), parses multi-format dates, and removes duplicate orders.
* **Logic & Transformations:**
  1. Filters `order_qty.isNotNull() & (order_qty > 0) & order_id.isNotNull()`.
  2. Strips leading weekday prefixes via `regexp_replace(order_placement_date, '^[A-Za-z]+,\\s*', '')`.
  3. Parses multi-format date strings (`yyyy/MM/dd`, `dd-MM-yyyy`, `dd/MM/yyyy`, `MMMM dd, yyyy`) into ISO DateType via `coalesce()`.
  4. Applies `dropDuplicates(['order_id', 'order_placement_date', 'customer_id', 'product_id', 'order_qty'])`.
* **Inputs & Dependencies:** Bronze DataFrame `df_orders`.
* **Outputs & Medallion State:** Cleaned DataFrame `df_orders` with valid dates and positive order quantities.

In [9]:
# 1. Keep only rows where order_qty is present
df_orders = df_orders.filter(F.col("order_qty").isNotNull())


# 2. Clean customer_id → keep numeric, else set to 999999
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
     .otherwise("999999")
     .cast("string")
)

# 3. Remove weekday name from the date text
#    "Tuesday, July 01, 2025" → "July 01, 2025"
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

# 4. Parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
    )
)

# 5. Drop duplicates
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

# 5. convert product id to string
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))

### 📌 Step 11: Inspect Cleansed Date Boundaries
* **Purpose:** Inspects minimum and maximum order placement dates to verify the temporal span of the backfill dataset.
* **Logic & Transformations:** Calculates `min('order_placement_date')` and `max('order_placement_date')` and renders output.
* **Inputs & Dependencies:** Cleaned DataFrame `df_orders`.
* **Outputs & Medallion State:** Minimum and maximum backfill dates displayed.

In [10]:
# check what's the maximum and minimum date
df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2025-07-01|2025-08-20|
+----------+----------+



### 📌 Step 12: Broadcast Join with Product Master to Attach Surrogate Key
* **Purpose:** Enriches transactional orders by joining with `fmcg.silver.products` on natural key `product_id` to resolve the conformed surrogate key `product_code`.
* **Logic & Transformations:** Performs an optimized broadcast join `df_orders.join(broadcast(df_products), 'product_id', 'left')`.
* **Inputs & Dependencies:** Orders DataFrame `df_orders` and Silver products table `fmcg.silver.products`.
* **Outputs & Medallion State:** Enriched DataFrame `df_joined` containing `product_code` alongside order details.

In [11]:
# Broadcast small dimension table to prevent cluster-wide shuffle join
from pyspark.sql.functions import broadcast
df_products = spark.table(f"{catalog}.{silver_schema}.products")
df_joined = df_orders.join(broadcast(df_products), on="product_id", how="inner").select(df_orders["*"], df_products["product_code"])
df_joined.show(5)


[Local Spark Emulation] spark.table adapted: fmcg.silver.products -> silver.products


+--------+--------------------+-----------+----------+---------+--------------------+--------------------+---------------+---------+------------+
|order_id|order_placement_date|customer_id|product_id|order_qty|           _metadata|      read_timestamp|      file_name|file_size|product_code|
+--------+--------------------+-----------+----------+---------+--------------------+--------------------+---------------+---------+------------+
|  ORD001|          2025-07-01|       1001|      P101|       50|{sample_data.csv,...|2026-09-17 12:43:...|sample_data.csv|     1024|   P101_CODE|
|  ORD002|                NULL|       1002|      P102|       20|{sample_data.csv,...|2026-09-17 12:43:...|sample_data.csv|     1024|   P102_CODE|
|  ORD003|          2025-08-01|       1003|      P103|      100|{sample_data.csv,...|2026-09-17 12:43:...|sample_data.csv|     1024|   P103_CODE|
|  ORD004|          2025-08-20|       1001|      P104|       15|{sample_data.csv,...|2026-09-17 12:43:...|sample_data.csv|  

### 📌 Step 13: Persist Cleansed Orders into Silver Delta Table
* **Purpose:** Writes the conformed transactional fact records into Silver Delta Lake using an ACID merge on composite natural keys.
* **Logic & Transformations:** If the table does not exist, creates it in `overwrite` mode with CDF enabled; otherwise, merges on `silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id`.
* **Inputs & Dependencies:** Enriched DataFrame `df_joined`.
* **Outputs & Medallion State:** Silver Delta table `fmcg.silver.orders` updated with validated order transactions.

In [12]:
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"), "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

[Local Spark Emulation] Adapted target table: fmcg.silver.orders -> silver.orders


26/09/17 12:44:12 WARN CreateNamespaceExec: Namespace silver was created concurrently. Ignoring.


### 📌 Step 14: Project Attributes for Subsidiary Gold Fact Layer
* **Purpose:** Extracts core transactional fields and standardizes column aliases (`order_placement_date -> date`, `customer_id -> customer_code`, `order_qty -> sold_quantity`).
* **Logic & Transformations:** Executes SQL projection against `{silver_table}` and previews the first 5 records.
* **Inputs & Dependencies:** Silver orders table `fmcg.silver.orders`.
* **Outputs & Medallion State:** Gold daily transactional DataFrame `df_gold`.

In [13]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {silver_table};")

df_gold.show(2)

[Local Spark Emulation] Multi-part namespace adapted: SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM fmcg.silver.orders; -> SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM silver.orders;


+--------+----------+-------------+------------+----------+-------------+
|order_id|      date|customer_code|product_code|product_id|sold_quantity|
+--------+----------+-------------+------------+----------+-------------+
|  ORD001|2025-07-01|         1001|   P101_CODE|      P101|           50|
|  ORD002|      NULL|         1002|   P102_CODE|      P102|           20|
+--------+----------+-------------+------------+----------+-------------+
only showing top 2 rows


### 📌 Step 15: Persist Daily Subsidiary Gold Fact Table (`sb_fact_orders`)
* **Purpose:** Saves daily subsidiary sales orders into `fmcg.gold.sb_fact_orders` for operational reporting at daily transaction grain.
* **Logic & Transformations:** If table does not exist, creates it with CDF; otherwise, executes Delta Lake `merge()` on composite key `(date, order_id, product_code, customer_code)`.
* **Inputs & Dependencies:** Daily Gold DataFrame `df_gold`.
* **Outputs & Medallion State:** Gold Delta table `fmcg.gold.sb_fact_orders` committed on storage.

In [14]:
if not (spark.catalog.tableExists(gold_table)):
    print("creating New Table")
    df_gold.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("source").merge(df_gold.alias("gold"), "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

creating New Table
[Local Spark Emulation] Adapted target table: fmcg.gold.sb_fact_orders -> gold.sb_fact_orders


### 📌 Step 16: Read Subsidiary Daily Orders for Enterprise Rollup
* **Purpose:** Reads daily records from `sb_fact_orders` to prepare aggregated monthly sales figures matching corporate reporting standards.
* **Logic & Transformations:** Queries `SELECT date, product_code, customer_code, sold_quantity FROM {gold_table}`.
* **Inputs & Dependencies:** Gold subsidiary table `fmcg.gold.sb_fact_orders`.
* **Outputs & Medallion State:** DataFrame `df_child` loaded into session scope.

In [15]:
df_child = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity FROM {gold_table}")
df_child.show(10)

[Local Spark Emulation] Multi-part namespace adapted: SELECT date, product_code, customer_code, sold_quantity FROM fmcg.gold.sb_fact_orders -> SELECT date, product_code, customer_code, sold_quantity FROM gold.sb_fact_orders
+----------+------------+-------------+-------------+
|      date|product_code|customer_code|sold_quantity|
+----------+------------+-------------+-------------+
|2025-07-01|   P101_CODE|         1001|           50|
|      NULL|   P102_CODE|         1002|           20|
|2025-08-01|   P103_CODE|         1003|          100|
|2025-08-20|   P104_CODE|         1001|           15|
+----------+------------+-------------+-------------+



### 📌 Step 17: Count Daily Transactional Fact Volume
* **Purpose:** Verifies total daily transaction volume before monthly aggregation.
* **Logic & Transformations:** Calls `df_child.count()`.
* **Inputs & Dependencies:** DataFrame `df_child`.
* **Outputs & Medallion State:** Record count printed to cell output.

In [16]:
df_child.count()

4

### 📌 Step 18: Aggregate Daily Orders to Monthly Enterprise Grain
* **Purpose:** Rolls up daily sales orders to monthly grain `(date, product_code, customer_code)` required by the enterprise parent star schema.
* **Logic & Transformations:**
  1. Truncates transaction date to month-start: `month_start = F.trunc('date', 'MM')`.
  2. Groups by `(month_start, product_code, customer_code)`.
  3. Aggregates `sold_quantity = F.sum('sold_quantity')`.
  4. Renames `month_start` back to `date`.
* **Inputs & Dependencies:** Daily DataFrame `df_child`.
* **Outputs & Medallion State:** Aggregated monthly DataFrame `df_monthly` at grain `(date, product_code, customer_code)`.

In [17]:
df_monthly = (
    df_child
    # 1. Get month start date (e.g., 2025-11-30 → 2025-11-01)
    .withColumn("month_start", F.trunc("date", "MM"))   # or F.date_trunc("month", "date").cast("date")

    # 2.Group at monthly grain by month_start + product_code + customer_code
    .groupBy("month_start", "product_code", "customer_code")
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )

    # 3. Rename month_start back to `date` to match your target schema
    .withColumnRenamed("month_start", "date")
)

df_monthly.show(5, truncate=False)

+----------+------------+-------------+-------------+
|date      |product_code|customer_code|sold_quantity|
+----------+------------+-------------+-------------+
|2025-07-01|P101_CODE   |1001         |50           |
|2025-08-01|P103_CODE   |1003         |100          |
|2025-08-01|P104_CODE   |1001         |15           |
|NULL      |P102_CODE   |1002         |20           |
+----------+------------+-------------+-------------+



### 📌 Step 19: Count Monthly Aggregated Fact Volume
* **Purpose:** Confirms record volume of the rolled-up monthly sales dataset.
* **Logic & Transformations:** Calls `df_monthly.count()`.
* **Inputs & Dependencies:** Aggregated DataFrame `df_monthly`.
* **Outputs & Medallion State:** Monthly row count printed to cell output.

In [18]:
df_monthly.count()

4

### 📌 Step 20: Data Quality Validation & Parent Fact Merge (`fact_orders`)
* **Purpose:** Validates monthly fact quality rules and executes an ACID Delta Lake merge into enterprise parent fact table `fmcg.gold.fact_orders`.
* **Logic & Transformations:**
  1. Runs quality check: `sold_quantity >= 0` via `run_quality_checks()`.
  2. Merges on composite primary key: `parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code`.
  3. Updates matching records and inserts new monthly product-customer combinations.
* **Inputs & Dependencies:** Aggregated DataFrame `df_monthly` and target Delta table `fmcg.gold.fact_orders`.
* **Outputs & Medallion State:** Enterprise parent fact table `fmcg.gold.fact_orders` updated with backfilled historical sales.

In [19]:
# Data Quality verification before parent merge
run_quality_checks(df_monthly, {"sold_qty_positive": ("sold_quantity >= 0", True)}, "fact_orders")

gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold").merge(
    df_monthly.alias("child_gold"),
    "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
print("Successfully merged monthly backfill into parent fact_orders")


[Local Spark Emulation] DeltaTable.forName adapted: fmcg.gold.fact_orders -> gold.fact_orders


26/09/17 12:44:31 WARN CreateNamespaceExec: Namespace gold was created concurrently. Ignoring.


Successfully merged monthly backfill into parent fact_orders


26/09/17 12:44:38 WARN MapPartitionsRDD: RDD 356 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


### 📌 Step 21: Lakehouse Maintenance: File Compaction & Multi-Dimensional Z-Ordering
* **Purpose:** Compacts small Parquet files and co-locates data on storage by primary query filter columns (`date`, `product_code`, `customer_code`) for blazing-fast BI query performance.
* **Logic & Transformations:** Executes `OPTIMIZE {catalog}.{gold_schema}.fact_orders ZORDER BY (date, product_code, customer_code)`.
* **Inputs & Dependencies:** Target Delta table `fmcg.gold.fact_orders`.
* **Outputs & Medallion State:** Optimized Delta storage layout with Z-Order indexing applied.

In [20]:
# Compaction & Z-ORDER optimization by primary query filter columns
spark.sql(f"OPTIMIZE {catalog}.{gold_schema}.fact_orders ZORDER BY (date, product_code, customer_code)")


[Local Spark Emulation] Multi-part namespace adapted: OPTIMIZE fmcg.gold.fact_orders ZORDER BY (date, product_code, customer_code) -> OPTIMIZE gold.fact_orders ZORDER BY (date, product_code, customer_code)


DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,